# feat/data-ingestion — Pipeline d'ingestion Steam

Notebook couvrant les 4 étapes de la branche :
1. Télécharger le dataset Kaggle
2. Nettoyer les données
3. Stocker dans MongoDB
4. Valider et documenter

> **Config centrale** : toutes les variables (chemins, URIs) viennent de `config.py` à la racine du projet.

## 1. Dépendances et configuration

In [ ]:
import sys, os
from pathlib import Path

# Ajouter la racine du projet au path pour importer config.py
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from pymongo import MongoClient, errors as mongo_errors
from kaggle.api.kaggle_api_extended import KaggleApi
import shutil, logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")

# Charger la config centrale
from config import (
    KAGGLE_DATASET,
    RAW_DATA_DIR, PROCESSED_DATA_DIR, FEATURES_DATA_DIR,
    GAMES_CSV, GAMES_JSON, CLEAN_CSV,
    MONGO_URI, MONGO_DB, MONGO_COLL,
)

print(f"Projet root  : {PROJECT_ROOT}")
print(f"RAW_DATA_DIR : {RAW_DATA_DIR}")
print(f"CLEAN_CSV    : {CLEAN_CSV}")
print(f"Kaggle       : {KAGGLE_DATASET}")
print(f"MongoDB      : {MONGO_URI}  db={MONGO_DB}  coll={MONGO_COLL}")

## 2. Préparer les répertoires dans `/data`

In [ ]:
for d in [RAW_DATA_DIR, PROCESSED_DATA_DIR, FEATURES_DATA_DIR]:
    d.mkdir(parents=True, exist_ok=True)
    print(f"OK  {d}")

# Espace disque sur /data
total, used, free = shutil.disk_usage("/data")
print(f"\nEspace /data  total={total//2**30}G  utilisé={used//2**30}G  libre={free//2**30}G")

## 3. Télécharger le dataset Kaggle dans `/data/raw`

In [ ]:
# Passer False pour forcer le re-téléchargement même si le fichier existe déjà
FORCE_DOWNLOAD = False

if FORCE_DOWNLOAD or not GAMES_CSV.exists():
    api = KaggleApi()
    api.authenticate()
    logging.info(f"Téléchargement : {KAGGLE_DATASET}")
    api.dataset_download_files(
        KAGGLE_DATASET,
        path=str(RAW_DATA_DIR),
        unzip=True,
        force=FORCE_DOWNLOAD,
        quiet=False,
    )
    logging.info("Téléchargement terminé.")
else:
    print("Fichiers déjà présents, téléchargement ignoré.")

print("\nFichiers dans RAW_DATA_DIR :")
for f in sorted(RAW_DATA_DIR.iterdir()):
    print(f"  {f.name}  ({f.stat().st_size // 2**20} Mo)")

## 4. Charger les fichiers bruts et profiler les colonnes

In [ ]:
df_raw = pd.read_csv(GAMES_CSV, low_memory=False)
print(f"Dimensions : {df_raw.shape[0]:,} lignes × {df_raw.shape[1]} colonnes\n")

# Types et valeurs manquantes
missing = df_raw.isnull().sum()
profile = pd.DataFrame({
    "dtype":   df_raw.dtypes,
    "missing": missing,
    "pct_missing": (missing / len(df_raw) * 100).round(2),
    "nunique": df_raw.nunique(),
})
print(profile.sort_values("pct_missing", ascending=False).head(20).to_string())

print(f"\nDoublons (AppID) : {df_raw.duplicated(subset=['AppID']).sum()}")

## 5. Nettoyer et normaliser les données

In [ ]:
def clean_games(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # 1. Renommer en snake_case
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(r"[ \-/]", "_", regex=True)
    )

    # 2. Supprimer les doublons sur l'identifiant Steam
    df = df.drop_duplicates(subset=["appid"])

    # 3. Supprimer les lignes sans nom
    df = df.dropna(subset=["name"])
    df["name"] = df["name"].str.strip()
    df = df[df["name"] != ""]

    # 4. Types numériques
    for col in ["price", "positive", "negative", "recommendations",
                "peak_ccu", "estimated_owners"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # 5. Score positif dérivé (évite division par zéro)
    if {"positive", "negative"}.issubset(df.columns):
        total = df["positive"] + df["negative"]
        df["positive_ratio"] = (df["positive"] / total.replace(0, pd.NA)).round(4)

    # 6. Champs liste (genres, categories, tags) : str → list
    for col in ["genres", "categories", "tags"]:
        if col in df.columns:
            df[col] = df[col].fillna("").apply(
                lambda v: [x.strip() for x in str(v).split(",") if x.strip()]
            )

    return df


df_clean = clean_games(df_raw)
before, after = len(df_raw), len(df_clean)
print(f"Lignes avant : {before:,}  →  après : {after:,}  (supprimées : {before-after:,})")
df_clean.head(3)

## 6. Valider la qualité des données nettoyées

In [ ]:
# Assertions métier
assert df_clean["appid"].is_unique, "appid non unique !"
assert df_clean["name"].notna().all(), "name contient des NaN !"
assert (df_clean["price"].fillna(0) >= 0).all(), "prix négatif détecté !"

# Taux de suppression
pct_removed = (before - after) / before * 100
print(f"Taux de suppression : {pct_removed:.1f}%")

# Distribution prix
if "price" in df_clean.columns:
    print("\nDistribution du prix (USD) :")
    print(df_clean["price"].describe().round(2))

print("\nAucune assertion n'a échoué ✓")

## 7. Transformer les enregistrements pour MongoDB

In [ ]:
import math

def to_mongo_doc(row: dict) -> dict:
    """Convertit une ligne pandas en document MongoDB propre."""
    doc = {}
    for k, v in row.items():
        # Remplacer les NaN/Inf par None (non sérialisable BSON sinon)
        if isinstance(v, float) and (math.isnan(v) or math.isinf(v)):
            doc[k] = None
        else:
            doc[k] = v
    # Utiliser appid comme _id Mongo
    doc["_id"] = int(doc.pop("appid"))
    return doc


records = df_clean.to_dict(orient="records")
documents = [to_mongo_doc(r) for r in records]

BATCH_SIZE = 500
batches = [documents[i:i + BATCH_SIZE] for i in range(0, len(documents), BATCH_SIZE)]
print(f"{len(documents):,} documents prêts en {len(batches)} batches de {BATCH_SIZE}")

## 8. Insérer les données dans MongoDB

In [ ]:
client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=5000)
# Vérification connexion
client.admin.command("ping")
print(f"Connexion MongoDB OK  →  {MONGO_URI}")

db   = client[MONGO_DB]
coll = db[MONGO_COLL]

# Vider la collection avant réinsertion (idempotent)
deleted = coll.delete_many({})
print(f"Collection nettoyée : {deleted.deleted_count} docs supprimés")

inserted_total = 0
errors_total   = 0

for i, batch in enumerate(batches):
    try:
        result = coll.insert_many(batch, ordered=False)
        inserted_total += len(result.inserted_ids)
    except mongo_errors.BulkWriteError as bwe:
        inserted_total += bwe.details.get("nInserted", 0)
        errors_total   += len(bwe.details.get("writeErrors", []))
    if (i + 1) % 10 == 0 or (i + 1) == len(batches):
        print(f"  Batch {i+1}/{len(batches)}  insérés={inserted_total:,}  erreurs={errors_total}")

print(f"\nInsertion terminée : {inserted_total:,} documents insérés, {errors_total} erreurs")
client.close()

## 9. Vérifier les insertions et sauvegarder les artefacts

In [ ]:
client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=5000)
coll   = client[MONGO_DB][MONGO_COLL]

count = coll.count_documents({})
print(f"Documents dans MongoDB ({MONGO_DB}.{MONGO_COLL}) : {count:,}")

print("\nÉchantillon (3 docs) :")
for doc in coll.find({}, {"_id": 1, "name": 1, "price": 1, "genres": 1}).limit(3):
    print(f"  {doc}")

client.close()

# Sauvegarder le CSV nettoyé dans /data/processed
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
df_clean.to_csv(CLEAN_CSV, index=False)
size_mb = CLEAN_CSV.stat().st_size / 2**20
print(f"\nCSV nettoyé sauvegardé : {CLEAN_CSV}  ({size_mb:.1f} Mo)")
print("\n✓ Pipeline data-ingestion terminé.")